# RAG

In [16]:
import ast

results_string = """
[Document(metadata={'creationdate': '', 'title': 'Resume', 'source': 'docs\\resume.pdf', 'page': 0, 'creator': 'Microsoft Office Word', 'producer': 'Aspose.Words for .NET 20.4', 'page_label': '1', 'total_pages': 1}, page_content='ADDITIONAL\n\uf0b7Certifications: AWS Machine Learning Engineer Associate, AWS AI Practitioner.\n\uf0b7Language Skills: Indonesian (Native), English (IELTS 7.5/9.0), Chinese (HSK 4: 205/300).\n\uf0b7Technical Skills: Python, C, SQL, R, Java.\n\uf0b7Volunteer Work: Math Video Translator at BINUS Library (2023), Junior ML Engineer at Omdena (2021).\n\uf0b7Work Authorization: Singapore (Internship), Indonesia (Citizen).'), Document(metadata={'title': 'Resume', 'creationdate': '', 'total_pages': 1, 'source': 'docs\\resume.pdf', 'page_label': '1', 'creator': 'Microsoft Office Word', 'page': 0, 'producer': 'Aspose.Words for .NET 20.4'}, page_content='ADDITIONAL\n\uf0b7Certifications: AWS Machine Learning Engineer Associate, AWS AI Practitioner.\n\uf0b7Language Skills: Indonesian (Native), English (IELTS 7.5/9.0), Chinese (HSK 4: 205/300).\n\uf0b7Technical Skills: Python, C, SQL, R, Java.\n\uf0b7Volunteer Work: Math Video Translator at BINUS Library (2023), Junior ML Engineer at Omdena (2021).\n\uf0b7Work Authorization: Singapore (Internship), Indonesia (Citizen).'), Document(metadata={'creationdate': '', 'source': 'docs\\resume.pdf', 'page': 0, 'creator': 'Microsoft Office Word', 'producer': 'Aspose.Words for .NET 20.4', 'page_label': '1', 'title': 'Resume', 'total_pages': 1}, page_content="\uf0b7Structured and gathered relevant keywords from 1000+ CVs for a faster candidate screening.\nPACMANN AI - Jakarta, Indonesia Aug 2022 - Feb 2023\nAI and data science bootcamp that prepares students for career switching and corporate upskilling.\nMachine Learning Teaching Assistant\n\uf0b7Conducted machine learning tutorials and obtained >3.5/4 teaching performance score.\n\uf0b7Moderated live-class operations, initiated weekly discussion, and graded students' assignments to support")]
"""
results_string

'\n[Document(metadata={\'creationdate\': \'\', \'title\': \'Resume\', \'source\': \'docs\\resume.pdf\', \'page\': 0, \'creator\': \'Microsoft Office Word\', \'producer\': \'Aspose.Words for .NET 20.4\', \'page_label\': \'1\', \'total_pages\': 1}, page_content=\'ADDITIONAL\n\uf0b7Certifications: AWS Machine Learning Engineer Associate, AWS AI Practitioner.\n\uf0b7Language Skills: Indonesian (Native), English (IELTS 7.5/9.0), Chinese (HSK 4: 205/300).\n\uf0b7Technical Skills: Python, C, SQL, R, Java.\n\uf0b7Volunteer Work: Math Video Translator at BINUS Library (2023), Junior ML Engineer at Omdena (2021).\n\uf0b7Work Authorization: Singapore (Internship), Indonesia (Citizen).\'), Document(metadata={\'title\': \'Resume\', \'creationdate\': \'\', \'total_pages\': 1, \'source\': \'docs\\resume.pdf\', \'page_label\': \'1\', \'creator\': \'Microsoft Office Word\', \'page\': 0, \'producer\': \'Aspose.Words for .NET 20.4\'}, page_content=\'ADDITIONAL\n\uf0b7Certifications: AWS Machine Learning 

In [21]:
import re

sections = re.split(r'\n(Education|Experience|Additional)\n', results_string)
sections

['\n[Document(metadata={\'creationdate\': \'\', \'title\': \'Resume\', \'source\': \'docs\\resume.pdf\', \'page\': 0, \'creator\': \'Microsoft Office Word\', \'producer\': \'Aspose.Words for .NET 20.4\', \'page_label\': \'1\', \'total_pages\': 1}, page_content=\'ADDITIONAL\n\uf0b7Certifications: AWS Machine Learning Engineer Associate, AWS AI Practitioner.\n\uf0b7Language Skills: Indonesian (Native), English (IELTS 7.5/9.0), Chinese (HSK 4: 205/300).\n\uf0b7Technical Skills: Python, C, SQL, R, Java.\n\uf0b7Volunteer Work: Math Video Translator at BINUS Library (2023), Junior ML Engineer at Omdena (2021).\n\uf0b7Work Authorization: Singapore (Internship), Indonesia (Citizen).\'), Document(metadata={\'title\': \'Resume\', \'creationdate\': \'\', \'total_pages\': 1, \'source\': \'docs\\resume.pdf\', \'page_label\': \'1\', \'creator\': \'Microsoft Office Word\', \'page\': 0, \'producer\': \'Aspose.Words for .NET 20.4\'}, page_content=\'ADDITIONAL\n\uf0b7Certifications: AWS Machine Learning

In [9]:
from datetime import datetime, timezone, timedelta

def get_current_timestamp():
    utc_now = datetime.now(timezone.utc)
    utc_plus_8 = timezone(timedelta(hours=8))
    time_in_utc_plus_8 = utc_now.astimezone(utc_plus_8)
    return time_in_utc_plus_8


In [10]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
import os
from dotenv import load_dotenv

embedding_dir = 'intellijob_embedding_db'
def embed_documents(root_folder='docs'):
    docs = os.listdir(root_folder)
    print(docs)

    all_chunks = []
    documents = []
    for doc in docs:
        file_path = os.path.join(root_folder, doc)
        pdf_loader = PyPDFLoader(file_path)
        document = pdf_loader.load()
        documents.extend(document)

        print(f'documents: {documents}')

        splitter = RecursiveCharacterTextSplitter(
            separators=['\n\n', '\n'],
            chunk_size=300,
            chunk_overlap=50
        )
        chunks = splitter.split_documents(document)
        print(f'chunks: {chunks}')

        now = get_current_timestamp()
        for idx, chunk in enumerate(chunks):
            chunk.metadata.update({
                'chunk_id': idx,
                'creation_date': now.strftime('%Y-%m-%d %H:%M:%S'),
            })

        all_chunks.extend(chunks)

    embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
    vector_store = Chroma.from_documents(
        documents=all_chunks,
        embedding=embeddings,
        persist_directory=embedding_dir,
        collection_name='personal_docs'
    )

    print(f'Added {len(all_chunks)} chunks to vector database.')
    return vector_store, chunks

In [11]:
now = get_current_timestamp()
now

datetime.datetime(2025, 9, 28, 16, 20, 21, 266909, tzinfo=datetime.timezone(datetime.timedelta(seconds=28800)))

In [13]:
vector_store, chunks = embed_documents('C:/Users/Dody Harianto/Documents/Personal Projects/LLM Projects/intellijob/docs')

['portfolio.pdf', 'resume.pdf']
documents: [Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2024-05-18T08:39:04+00:00', 'title': 'Projects Portfolio - Dody Harianto', 'moddate': '2024-05-18T08:39:00+00:00', 'keywords': 'DAFd1kEvsco,BAEPmQqx-To', 'author': 'Dody Harianto', 'source': 'C:/Users/Dody Harianto/Documents/Personal Projects/LLM Projects/intellijob/docs\\portfolio.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Dody Harianto\nProjects Portfolio\nhariantodody14@gmail.com\nhttps://www.linkedin.com/in/dodyharianto/\n+62 895 3391 31039\nhttps://github.com/dodyharianto\nhttps://www.kaggle.com/dodyharianto'), Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2024-05-18T08:39:04+00:00', 'title': 'Projects Portfolio - Dody Harianto', 'moddate': '2024-05-18T08:39:00+00:00', 'keywords': 'DAFd1kEvsco,BAEPmQqx-To', 'author': 'Dody Harianto', 'source': 'C:/Users/Dody Harianto/Documents/Personal Projects/LLM Projec

In [34]:
chunks

[Document(metadata={'producer': 'Aspose.Words for .NET 20.4', 'creator': 'Microsoft Office Word', 'creationdate': '', 'title': 'Resume', 'source': 'docs\\resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'chunk_id': 0, 'creation_date': '2025-09-26 21:05:20'}, page_content='DODY HARIANTO\nd.harianto.2025@mitb.smu.edu.sg +65 8967 1350 https://linktr.ee/dodyharianto\nEDUCATION\nSINGAPORE MANAGEMENT UNIVERSITY (SMU) Aug 2025 - Dec 2026\nMaster of IT in Business (Artificial Intelligence)\nBINA NUSANTARA UNIVERSITY (BINUS)- Jakarta, Indonesia Sep 2020 - Apr 2025'),
 Document(metadata={'producer': 'Aspose.Words for .NET 20.4', 'creator': 'Microsoft Office Word', 'creationdate': '', 'title': 'Resume', 'source': 'docs\\resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'chunk_id': 1, 'creation_date': '2025-09-26 21:05:20'}, page_content='Bachelor of Science in Computer Science and Mathematics\n\uf0b7Top 10 Finalist of FIND IT! Gadjah Mada University Data Analytics Competiti

In [ ]:
sample_chunk = chunks[0]
sample_chunk.metadata

'Resume'

# Check Embedded Results

In [3]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()
embedding_dir = os.environ.get("EMBEDDING_DB_DIR")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory=embedding_dir,
    collection_name='personal_docs'
)

C:\Users\Dody Harianto\AppData\Local\Temp\ipykernel_74588\3324700555.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_store = Chroma(


In [9]:
docs = vector_store.get()
docs

{'ids': ['98e1adc5-8a18-4c18-9225-22babdb568ca',
  'f203be26-e150-4a52-bee6-34475a1986c8',
  '6745475d-0fbd-4f0d-8aaf-191bf3dca2a2',
  '841da30a-97f8-42ef-8809-c117dfb52571',
  '6cc787f8-685d-4126-9ba9-20a1a3683a07',
  'cee56e10-e02e-44fc-abf0-83433c186a9e',
  '94f6d1ce-45c6-4a09-bc48-822054da1324',
  '0eb2307f-cf85-4c45-91f8-043ccc130773',
  '2d506ea4-6acb-4158-af07-f33664cee208',
  '51db87eb-9b5f-443f-beec-32e4faddbdac',
  '12fcfd0a-200d-4d36-8d9e-ba8609447b1b',
  '69d448af-a421-48d0-929a-0ebed3f5931f',
  'a895bf9c-2200-42f3-8de5-97c14edda681',
  'cfbb871e-1f4e-4c7f-b848-b42251f35087',
  'a9997027-c0d0-48d5-bc5b-576b32a5d3f6',
  'ed8da01d-bfb6-4209-a3ee-563d69f84317',
  '0288a14b-5848-453a-a546-1f590cdeef18',
  '5c2ee9ec-1b67-4abb-9cd8-4a628623e004',
  '5f449477-ef43-48d9-8c52-e00ceb99396b',
  '072a7b32-c3b6-4283-999e-27cd09a8ddee',
  'db178352-5a80-4b27-84e9-078acbef53d9',
  '6f74a92e-9ca2-4ac3-b7dc-ec28d5bf6e84',
  '2f1c897f-1164-4780-be59-40f9d22e7cb1',
  '98618af2-d255-445b-99f1-

In [13]:
sorted(docs['documents'])

['Algorithm and Programming Mentor\n\uf0b7Taught algorithms with C programming to new university students to ensure smooth transition.\n\uf0b7Prepared competitive programming practice problems, demonstrated best teaching practices, and received 92% \npositive feedback (rating >3/5).\nADDITIONAL',
 'Algorithm and Programming Mentor\n\uf0b7Taught algorithms with C programming to new university students to ensure smooth transition.\n\uf0b7Prepared competitive programming practice problems, demonstrated best teaching practices, and received 92% \npositive feedback (rating >3/5).\nADDITIONAL',
 'Bachelor of Science in Computer Science and Mathematics\n\uf0b7Top 10 Finalist of FIND IT! Gadjah Mada University Data Analytics Competition 2022.\n\uf0b7University delegate for National Math and Sciences Competition (KNMIPA-PT) for Jakarta regional level 2021.\nEXPERIENCE',
 'Bachelor of Science in Computer Science and Mathematics\n\uf0b7Top 10 Finalist of FIND IT! Gadjah Mada University Data Analy

# Clear Collection

In [35]:
from chromadb import PersistentClient

client = PersistentClient(path='intellijob_embedding_db')
client

In [40]:
collection = client.get_collection(name='personal_docs')
collection

Collection(name=personal_docs)

In [41]:
len(collection.get()['ids'])

35

In [43]:
document_ids = collection.get()['ids']
document_ids

['4cf058bc-962f-4881-a8ae-1144a8b003ae',
 '66cd9fae-6c76-4437-a6a8-958290b35f19',
 '84cf9723-a3e5-4c94-959f-7241012ffa4f',
 '2a4f2702-095b-4e2a-b918-8d9582ae8b95',
 '7c001ec4-daae-4e1e-8a90-dca07f3ecc31',
 '5c3d4238-f1cf-4a11-894b-222b7224a143',
 '487f3894-e8d9-4334-be74-06289f8b4de9',
 '9558f6fe-cd0f-4ac0-8825-38fbeda4ce0f',
 '6ffe5774-7f1b-43eb-8d6c-320f536bc309',
 'fc8cad1c-8219-4762-83fd-bbab28ebc158',
 'eb255774-ff3c-4855-8d62-285192a511b1',
 '6616a574-ed62-4f32-91c6-e03efc194d36',
 '855fd347-38c6-492b-ba1a-85f884a7411a',
 'fbea613e-4eb4-457f-a1ef-7d28ca851a98',
 'cb1701d1-2880-4dfa-a17c-b8fd496d388f',
 '23008e4b-85d4-4d8c-a0c5-309dc71f76e4',
 '192657b6-72ab-402d-b573-f5107125c604',
 '5d14ff13-6fa8-4ac0-9637-17a83698a77f',
 '6e20b2a3-9a22-4dc5-958e-906af15678e6',
 '989d1b7f-a8f3-4cd0-b6b0-5fb8387f2429',
 '03c90eb4-4328-4c75-b159-8d46f578c216',
 '9537f768-de60-4224-a03a-bd33b6c0b45d',
 '0a4a3eb8-6fff-42e7-88ed-39d4932a28a5',
 '29fd85e9-95db-48b1-a0e7-5bf4b930d7ba',
 '0baa25e7-5300-

In [ ]:
collection.delete(ids=document_ids)

In [31]:
len(collection.get()['ids'])

0

All documents have been deleted.

# Chat History

In [29]:
import sqlite3

def get_chat_history(n_messages=5):
    conn = sqlite3.connect('chat_history.db')
    cursor = conn.cursor()

    # Get the last n messages
    cursor.execute("""
        SELECT role, message
        FROM chat_history
        ORDER BY id DESC
        LIMIT (?)
    """, (n_messages, ))

    last_n_chats = cursor.fetchall()
    conn.commit()
    conn.close()
    return last_n_chats[::-1]

history = get_chat_history(3)
print(history)

[('system', '\n    You are an expert in helping user finding the most relevant jobs based on their resume and portfolio.\n    '), ('user', 'what can you do?'), ('assistant', 'I can help you land the most relevant jobs by aligning your resume and portfolio with real openings. Here’s what I can do:\n\n- Resume and portfolio analysis\n  - Review your resume and portfolio to identify strengths, gaps, and how well you map to target roles.\n  - Extract keywords from job descriptions and ensure your materials match ATS and recruiter expectations.\n\n- Role matching and targeting\n  - Suggest the best-fit roles, industries, and seniority levels based on your experience.\n  - Create a prioritized target list (companies, titles, locations, remote options).\n\n- Tailoring and optimization\n  - Rewrite resume bullets to emphasize impact, metrics, and transferable skills.\n  - Tailor your resume for 1–3 specific jobs, including ATS-friendly formatting and strong action verbs.\n  - Improve portfolio

In [33]:
history[0][1].strip()

'You are an expert in helping user finding the most relevant jobs based on their resume and portfolio.'

Run the following to delete rows in sqlite3.

In [ ]:
# conn = sqlite3.connect('chat_history.db')
# cursor = conn.cursor()

# # Get the last n messages
# cursor.execute("""
#     DELETE FROM chat_history
#     WHERE role = 'system'
# """)

# conn.commit()
# conn.close()

In [37]:
get_chat_history(5)

[('user', 'what can you do?'),
 ('assistant',
  'I can help you land the most relevant jobs by aligning your resume and portfolio with real openings. Here’s what I can do:\n\n- Resume and portfolio analysis\n  - Review your resume and portfolio to identify strengths, gaps, and how well you map to target roles.\n  - Extract keywords from job descriptions and ensure your materials match ATS and recruiter expectations.\n\n- Role matching and targeting\n  - Suggest the best-fit roles, industries, and seniority levels based on your experience.\n  - Create a prioritized target list (companies, titles, locations, remote options).\n\n- Tailoring and optimization\n  - Rewrite resume bullets to emphasize impact, metrics, and transferable skills.\n  - Tailor your resume for 1–3 specific jobs, including ATS-friendly formatting and strong action verbs.\n  - Improve portfolio project descriptions (problem, approach, results, tools, and visuals).\n\n- Portfolio enhancements\n  - Audit your portfolio 

# Document Chunking with Metadata

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from chromadb import PersistentClient
import os
from dotenv import load_dotenv

In [3]:
EMBEDDING_DIR = os.environ.get("EMBEDDING_DB_DIR")
EMBEDDING_MODEL = OpenAIEmbeddings(model="text-embedding-3-small")

In [4]:
client = PersistentClient(path=EMBEDDING_DIR)
client

In [6]:
print(f'There are {client.count_collections()} collections.')

There are 0 collections.


In [19]:
collection = client.create_collection('test-collection')

In [21]:
client.list_collections()

[Collection(name=test-collection)]

In [23]:
client.delete_collection('test-collection')

Using `ChromaDB()`:

In [22]:
vector_store = Chroma(
    collection_name='user_profile',
    embedding_function=EMBEDDING_MODEL,
    persist_directory=EMBEDDING_DIR
)

vector_store

In [25]:
vector_store._collection

Collection(name=user_profile)

In [ ]:
def embed_documents(collection_name: str, file_path: str) -> List[DocumentChunk]:
    pdf_loader = PyPDFLoader(file_path)
    docs = pdf_loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100
    )
    chunks = splitter.split_documents(docs)
    return chunks

In [ ]:
chunks = embed_documents(file_path='../docs/resume.pdf', collection_name='user_profile')
chunks

[Document(metadata={'producer': 'Aspose.Words for .NET 20.4', 'creator': 'Microsoft Office Word', 'creationdate': '', 'title': 'Resume', 'source': '../docs/resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content="DODY HARIANTO\nd.harianto.2025@mitb.smu.edu.sg +65 8967 1350 https://linktr.ee/dodyharianto\nEDUCATION\nSINGAPORE MANAGEMENT UNIVERSITY (SMU) Aug 2025 - Dec 2026\nMaster of IT in Business (Artificial Intelligence)\nBINA NUSANTARA UNIVERSITY (BINUS)- Jakarta, Indonesia Sep 2020 - Apr 2025\nBachelor of Science in Computer Science and Mathematics\n\uf0b7Top 10 Finalist of FIND IT! Gadjah Mada University Data Analytics Competition 2022.\n\uf0b7University delegate for National Math and Sciences Competition (KNMIPA-PT) for Jakarta regional level 2021.\nEXPERIENCE\nTEMAN DATA - Jakarta, Indonesia Oct 2023 - Jul 2025\nTeman Data provides data analytics and AI services tailored to client's needs."),
 Document(metadata={'producer': 'Aspose.Words for .NET 20.4', 'creat

In [62]:
for idx, chunk in enumerate(chunks):
    print(f'Chunk {idx}')
    print(chunk.page_content)
    print('\n')

Chunk 0
Dody Harianto
Projects Portfolio
hariantodody14@gmail.com
https://www.linkedin.com/in/dodyharianto/
+62 895 3391 31039
https://github.com/dodyharianto
https://www.kaggle.com/dodyharianto


Chunk 1
Technology Stack


Chunk 2
This project shows the trends of vehicle
price in Australia based on brands,
number of car doors, transmission, and
car type, in form of an interactive
dashboard.
Australia Vehicle Price Analysis
PROJECT DESCRIPTION
Project Link
https://lookerstudio.google.com/reporting/2831a988-b46f-4d39-91a0-d03c2cb42c8a
TOOLS
Looker Studio, Python


Chunk 3
This project aims to show the statistics
of Indonesia’s export and import trend
in terms of value, commodity types,
comparison with other countries. 
Indonesia’s Last 10 YearsExport & Import Analysis
PROJECT DESCRIPTION
Project Link
https://dodyharianto-tetris-3-capstone-project-app-k6gx2y.streamlit.app/
TOOLS
Python, R, Streamlit, Tableau


Chunk 4
This project aims to extract text from PDF
files which can be used to 

In [ ]:
chunks = embed_documents(file_path='../docs/portfolio.pdf', collection_name='user_profile')
chunks

[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2024-05-18T08:39:04+00:00', 'title': 'Projects Portfolio - Dody Harianto', 'moddate': '2024-05-18T08:39:00+00:00', 'keywords': 'DAFd1kEvsco,BAEPmQqx-To', 'author': 'Dody Harianto', 'source': '../docs/portfolio.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Dody Harianto\nProjects Portfolio\nhariantodody14@gmail.com\nhttps://www.linkedin.com/in/dodyharianto/\n+62 895 3391 31039\nhttps://github.com/dodyharianto\nhttps://www.kaggle.com/dodyharianto'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2024-05-18T08:39:04+00:00', 'title': 'Projects Portfolio - Dody Harianto', 'moddate': '2024-05-18T08:39:00+00:00', 'keywords': 'DAFd1kEvsco,BAEPmQqx-To', 'author': 'Dody Harianto', 'source': '../docs/portfolio.pdf', 'total_pages': 12, 'page': 1, 'page_label': '2'}, page_content='Technology Stack'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creati

In [63]:
for idx, chunk in enumerate(chunks):
    print(f'Chunk {idx}')
    print(chunk.page_content)
    print('\n')

Chunk 0
Dody Harianto
Projects Portfolio
hariantodody14@gmail.com
https://www.linkedin.com/in/dodyharianto/
+62 895 3391 31039
https://github.com/dodyharianto
https://www.kaggle.com/dodyharianto


Chunk 1
Technology Stack


Chunk 2
This project shows the trends of vehicle
price in Australia based on brands,
number of car doors, transmission, and
car type, in form of an interactive
dashboard.
Australia Vehicle Price Analysis
PROJECT DESCRIPTION
Project Link
https://lookerstudio.google.com/reporting/2831a988-b46f-4d39-91a0-d03c2cb42c8a
TOOLS
Looker Studio, Python


Chunk 3
This project aims to show the statistics
of Indonesia’s export and import trend
in terms of value, commodity types,
comparison with other countries. 
Indonesia’s Last 10 YearsExport & Import Analysis
PROJECT DESCRIPTION
Project Link
https://dodyharianto-tetris-3-capstone-project-app-k6gx2y.streamlit.app/
TOOLS
Python, R, Streamlit, Tableau


Chunk 4
This project aims to extract text from PDF
files which can be used to 

In [66]:
chunks[1].metadata

{'producer': 'Canva',
 'creator': 'Canva',
 'creationdate': '2024-05-18T08:39:04+00:00',
 'title': 'Projects Portfolio - Dody Harianto',
 'moddate': '2024-05-18T08:39:00+00:00',
 'keywords': 'DAFd1kEvsco,BAEPmQqx-To',
 'author': 'Dody Harianto',
 'source': '../docs/portfolio.pdf',
 'total_pages': 12,
 'page': 1,
 'page_label': '2'}

In [64]:
chunks[0].page_content

'Dody Harianto\nProjects Portfolio\nhariantodody14@gmail.com\nhttps://www.linkedin.com/in/dodyharianto/\n+62 895 3391 31039\nhttps://github.com/dodyharianto\nhttps://www.kaggle.com/dodyharianto'

In [5]:
from pydantic import BaseModel
from typing import List, Optional

class DocumentChunk(BaseModel):
    id: int
    source: str
    content: str
    page_number: int

In [81]:
def get_vector_store(collection_name: str):
    return Chroma(
        collection_name=collection_name,
        embedding_function=EMBEDDING_MODEL,
        persist_directory=EMBEDDING_DIR
    )

In [82]:
def embed_documents(file_path: str, collection_name: str = 'user_profile') -> List[DocumentChunk]:
    pdf_loader = PyPDFLoader(file_path)
    docs = pdf_loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100
    )
    chunks = splitter.split_documents(docs)
    clean_chunks = []
    for idx, chunk in enumerate(chunks):
        clean_chunk = DocumentChunk(
            chunk_id=idx,
            source=chunk.metadata.get('source'),
            content=chunk.page_content,
            page_number=int(chunk.metadata.get('page_label'))
        )
        clean_chunks.append(clean_chunk)
    
    vector_store = get_vector_store(collection_name)
    vector_store.add_documents(clean_chunks)
    return clean_chunks

In [ ]:
# This code will result in an error:
# AttributeError: 'DocumentChunk' object has no attribute 'id'
# Because custom Pydantic object 'DocumentChunk' does not have id and vector_store.add_documents strictly expect LangChain Document, which has an id in it

# chunks = embed_documents(file_path='../docs/portfolio.pdf', collection_name='user_profile')
# chunks

In [85]:
from langchain_core.documents import Document

In [ ]:
def embed_documents(file_path: str, collection_name: str = 'user_profile') -> List[Document]:
    pdf_loader = PyPDFLoader(file_path)
    docs = pdf_loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100
    )
    chunks = splitter.split_documents(docs)

    vector_store = get_vector_store(collection_name)
    vector_store.add_documents(chunks)
    return chunks

In [87]:
chunks = embed_documents(file_path='../docs/portfolio.pdf', collection_name='user_profile')
chunks

[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2024-05-18T08:39:04+00:00', 'title': 'Projects Portfolio - Dody Harianto', 'moddate': '2024-05-18T08:39:00+00:00', 'keywords': 'DAFd1kEvsco,BAEPmQqx-To', 'author': 'Dody Harianto', 'source': '../docs/portfolio.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Dody Harianto\nProjects Portfolio\nhariantodody14@gmail.com\nhttps://www.linkedin.com/in/dodyharianto/\n+62 895 3391 31039\nhttps://github.com/dodyharianto\nhttps://www.kaggle.com/dodyharianto'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2024-05-18T08:39:04+00:00', 'title': 'Projects Portfolio - Dody Harianto', 'moddate': '2024-05-18T08:39:00+00:00', 'keywords': 'DAFd1kEvsco,BAEPmQqx-To', 'author': 'Dody Harianto', 'source': '../docs/portfolio.pdf', 'total_pages': 12, 'page': 1, 'page_label': '2'}, page_content='Technology Stack'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creati

# Embedding Multiple Files

In [26]:
import os
docs = os.listdir('../docs')
docs

['portfolio.pdf', 'resume.pdf']

In [ ]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import os
from dotenv import load_dotenv

load_dotenv()
EMBEDDING_DIR = os.environ.get("EMBEDDING_DB_DIR")
EMBEDDING_MODEL = OpenAIEmbeddings(model='text-embedding-3-small')

def get_vector_store(collection_name: str):
    return Chroma(
        collection_name=collection_name,
        embedding_function=EMBEDDING_MODEL,
        persist_directory=EMBEDDING_DIR
    )

def embed_documents(document_dir: str = '../docs', collection_name: str = 'user_profile') -> List[Document]:
    vector_store = get_vector_store(collection_name)
    existing_chunk_ids = vector_store.get()['ids']

    # Delete the previous ids to ensure no duplicates
    if existing_chunk_ids:
        vector_store.delete(ids=existing_chunk_ids)

    docs = os.listdir(document_dir)
    all_chunks = []
    for doc in docs:
        file_path = os.path.join(document_dir, doc)
        print(file_path)
        pdf_loader = PyPDFLoader(file_path)
        document = pdf_loader.load()

        splitter = RecursiveCharacterTextSplitter(
            separators=['\n\n', '\n'],
            chunk_size=1000,
            chunk_overlap=100
        )
        chunks = splitter.split_documents(document)
        vector_store.add_documents(chunks)
        all_chunks.extend(chunks)

    return all_chunks

def retrieve_chunks(query: str, k: int = 3, collection_name: str = 'user_profile') -> List[Document]:
    vector_store = get_vector_store(collection_name)
    results = vector_store.similarity_search(query, k=k)
    return results

In [57]:
vector_store = get_vector_store('user_profile')
existing_chunk_ids = vector_store.get()
len(existing_chunk_ids['ids'])

52

In [58]:
embed_documents()

../docs\portfolio.pdf
../docs\resume.pdf


[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2024-05-18T08:39:04+00:00', 'title': 'Projects Portfolio - Dody Harianto', 'moddate': '2024-05-18T08:39:00+00:00', 'keywords': 'DAFd1kEvsco,BAEPmQqx-To', 'author': 'Dody Harianto', 'source': '../docs\\portfolio.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Dody Harianto\nProjects Portfolio\nhariantodody14@gmail.com\nhttps://www.linkedin.com/in/dodyharianto/\n+62 895 3391 31039\nhttps://github.com/dodyharianto\nhttps://www.kaggle.com/dodyharianto'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2024-05-18T08:39:04+00:00', 'title': 'Projects Portfolio - Dody Harianto', 'moddate': '2024-05-18T08:39:00+00:00', 'keywords': 'DAFd1kEvsco,BAEPmQqx-To', 'author': 'Dody Harianto', 'source': '../docs\\portfolio.pdf', 'total_pages': 12, 'page': 1, 'page_label': '2'}, page_content='Technology Stack'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'crea

Creation date time shows 2024.

In [59]:
vector_store = get_vector_store('user_profile')
existing_chunk_ids = vector_store.get()
len(existing_chunk_ids['ids'])

16

In [68]:
query = 'where did i work?'
retrieved_chunks = retrieve_chunks(query)
for idx, chunk in enumerate(retrieved_chunks):
    print(f'\nChunk {idx + 1}: ')
    print(chunk.page_content)


Chunk 1: 
techniques, resulting in ~50% of reduced runtime, as a base for restaurant menu creation.
Developed an AI agent for analyzing CSV files, enhancing company's RAG document retrieval flexibility.
Utilized Microsoft Power Platform to automate 20+ business workflows such as email notifications, dashboard 
monitoring, digital document signing, meeting scheduling, and client project monitoring app.
Awarded Most Creative Employee (Q4 2024) for innovation and approaches in solving tasks.
Data Engineer Intern (Oct 2023 - Jan 2024)
Programmed 2 ETL pipelines to preprocess and store medical data in Snowflake data warehouse.
Created 90+ data integration connections from various hospital database sources using Airbyte.
TORCHE EDUCATION - Tangerang, Banten, Indonesia Nov 2022 - Jun 2023
E-learning provider that offers personalized engineering courses for university students.
Python Engineer Intern

Chunk 2: 
Taught algorithms with C programming to new university students to ensure sm

# Update Metadata: Creation Date

In [ ]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import os
from dotenv import load_dotenv
from datetime import datetime, timezone, timedelta

load_dotenv()
EMBEDDING_DIR = os.environ.get("EMBEDDING_DB_DIR")
EMBEDDING_MODEL = OpenAIEmbeddings(model='text-embedding-3-small')

def get_current_timestamp():
    utc_now = datetime.now(timezone.utc)
    utc_plus_8 = timezone(timedelta(hours=8))
    time_in_utc_plus_8 = utc_now.astimezone(utc_plus_8)
    return time_in_utc_plus_8

def get_vector_store(collection_name: str):
    return Chroma(
        collection_name=collection_name,
        embedding_function=EMBEDDING_MODEL,
        persist_directory=EMBEDDING_DIR
    )

def embed_documents(document_dir: str = '../docs', collection_name: str = 'user_profile') -> List[Document]:
    vector_store = get_vector_store(collection_name)
    existing_chunk_ids = vector_store.get()['ids']

    # Delete the previous ids to ensure no duplicates
    if existing_chunk_ids:
        vector_store.delete(ids=existing_chunk_ids)

    docs = os.listdir(document_dir)
    all_chunks = []
    for doc in docs:
        file_path = os.path.join(document_dir, doc)
        print(file_path)
        pdf_loader = PyPDFLoader(file_path)
        document = pdf_loader.load()

        splitter = RecursiveCharacterTextSplitter(
            separators=['\n\n', '\n'],
            chunk_size=1000,
            chunk_overlap=100
        )
        chunks = splitter.split_documents(document)
        now = get_current_timestamp()
        for idx, chunk in enumerate(chunks):
            chunk.metadata.update({
                'creationdate': now.strftime('%Y-%m-%d %H:%M:%S'),
                'moddate': now.strftime('%Y-%m-%d %H:%M:%S'),
            })
            
        vector_store.add_documents(chunks)
        all_chunks.extend(chunks)

    return all_chunks

def retrieve_chunks(query: str, k: int = 3, collection_name: str = 'user_profile') -> List[Document]:
    vector_store = get_vector_store(collection_name)
    results = vector_store.similarity_search(query, k=k)
    return results

In [78]:
embed_documents()

../docs\portfolio.pdf
../docs\resume.pdf


[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-01-06 16:36:01', 'title': 'Projects Portfolio - Dody Harianto', 'moddate': '2024-05-18T08:39:00+00:00', 'keywords': 'DAFd1kEvsco,BAEPmQqx-To', 'author': 'Dody Harianto', 'source': '../docs\\portfolio.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Dody Harianto\nProjects Portfolio\nhariantodody14@gmail.com\nhttps://www.linkedin.com/in/dodyharianto/\n+62 895 3391 31039\nhttps://github.com/dodyharianto\nhttps://www.kaggle.com/dodyharianto'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-01-06 16:36:01', 'title': 'Projects Portfolio - Dody Harianto', 'moddate': '2024-05-18T08:39:00+00:00', 'keywords': 'DAFd1kEvsco,BAEPmQqx-To', 'author': 'Dody Harianto', 'source': '../docs\\portfolio.pdf', 'total_pages': 12, 'page': 1, 'page_label': '2'}, page_content='Technology Stack'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '